In [ ]:
import torch
import onnx
from typing import Tuple, Optional


def _find_first_conv_in_channels(model: torch.nn.Module) -> Optional[int]:
    """Return in_channels of the first nn.Conv2d found in module traversal or None."""
    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            return m.in_channels
    return None

def export_model_to_onnx(
    model_path: str,
    onnx_path: str,
    input_shape: Optional[Tuple[int, int, int, int]] = None,
    opset_version: int = 11,
) -> None:
    """
    Export a PyTorch model to ONNX. If input_shape is not provided, try to infer it from
    the loaded model (attributes `input_channels`, `input_height`, `input_width`) or by
    inspecting the first Conv2d layer. This reduces failures caused by a mismatched dummy input.

    Args:
        model_path (str): Path to the saved PyTorch model (.pt).
        onnx_path (str): Path where the .onnx file will be written.
        input_shape (Optional[Tuple[int,int,int,int]]): Example input shape as (N,C,H,W).
        opset_version (int): ONNX opset version to use.

    Raises:
        AssertionError: If provided paths are invalid or the loaded object is not a torch.nn.Module.
        RuntimeError: If export fails. Detailed diagnostics will be printed.
    """
    assert model_path, 'model_path must be provided'
    assert onnx_path, 'onnx_path must be provided'

    # Load and prepare the model
    model = torch.load(model_path, map_location='cpu')
    assert isinstance(model, torch.nn.Module), (
        f'Loaded object from {model_path} is {type(model)}; expected a torch.nn.Module. '
        'If you saved only a state_dict, load it into the model class before exporting.'
    )
    model.eval()
    model.cpu()
    
    # print model summary
    print(f'Model loaded from {model_path}. Type: {type(model)}')
    if hasattr(model, 'summary'):
        model.summary()  # If the model has a summary method, call it to print details
    else:
        print('Model summary method not available. Skipping summary.')

    # Ensure model is in evaluation mode

    # Infer input shape if not provided
    if input_shape is None:
        c = getattr(model, 'input_channels', None)
        h = getattr(model, 'input_height', None)
        w = getattr(model, 'input_width', None)
        if c is None:
            c = _find_first_conv_in_channels(model)
        if c is None:
            # Last resort: common defaults tried in order
            for candidate in (3, 1, 2, 4, 8):
                c = candidate
                break
        if h is None or w is None:
            h = 256 if getattr(model, 'input_height', None) is None else model.input_height
            w = 256 if getattr(model, 'input_width', None) is None else model.input_width
        input_shape = (1, int(c), int(h), int(w))
        print(f'Inferred input_shape = {input_shape}')
    else:
        assert isinstance(input_shape, tuple) and len(input_shape) == 4, 'input_shape must be a 4-tuple (N,C,H,W)'

    # Create dummy input
    dummy_input = torch.randn(input_shape, dtype=torch.float32)

    # Sanity run to catch forward errors early and print diagnostics
    try:
        with torch.no_grad():
            output = model(dummy_input)
            print(f'Model forward succeeded. Output shape: {getattr(output, "shape", str(type(output)))}')
    except Exception as e:
        # Try to provide helpful diagnostics for channel/shape mismatches
        print('Model forward failed with the inferred dummy input. Gathering diagnostics...')
        try:
            if hasattr(model, 'encoder_forward'):
                feats = model.encoder_forward(dummy_input, features_only=True)
                print('Encoder feature shapes:')
                for i, f in enumerate(feats):
                    print(f'  feat[{i}].shape = {f.shape}')
        except Exception as diag_exc:
            print(f'Failed to run encoder_forward for diagnostics: {diag_exc}')
        raise RuntimeError(f'Model forward pass failed with dummy input: {e}')

    # Attempt to export using scripting first (more robust) then tracing
    try:
        with torch.no_grad():
            scripted_model = torch.jit.script(model)
            export_model = scripted_model
            print('Exporting using torch.jit.script.')
    except Exception as script_error:
        print(f'Scripting failed: {script_error}. Falling back to tracing.')
        try:
            with torch.no_grad():
                traced_model = torch.jit.trace(model, dummy_input, strict=False)
                export_model = traced_model
                print('Exporting using torch.jit.trace.')
        except Exception as trace_error:
            raise RuntimeError(
                f'Both scripting and tracing failed. Scripting error: {script_error}. '
                f'Tracing error: {trace_error}.'
            )

    # Export the model to ONNX
    try:
        with torch.no_grad():
            torch.onnx.export(
                export_model,
                dummy_input,
                onnx_path,
                export_params=True,
                opset_version=opset_version,
                do_constant_folding=True,
                input_names=['input'],
                output_names=['output'],
                dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
                training=torch.onnx.TrainingMode.EVAL,
                verbose=True
            )
        print(f'Model successfully exported to ONNX at {onnx_path}.')
    except Exception as export_error:
        raise RuntimeError(f'Failed to export the model to ONNX: {export_error}')

    # Validate the ONNX model
    try:
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        print(f'ONNX model at {onnx_path} is valid.')
    except Exception as validation_error:
        raise RuntimeError(f'ONNX model validation failed: {validation_error}')


# Example usage (leave input_shape=None to auto-detect)
model_path = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/notebooks/onnx/model_and_architecture_26.pt'
onnx_path = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/notebooks/onnx/model_and_architecture_26.onnx'
export_model_to_onnx(model_path, onnx_path, input_shape=(1,2,1200,1200))

Model loaded from /Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/notebooks/onnx/model_and_architecture_26.pt. Type: <class 'torch.jit._script.RecursiveScriptModule'>
Model summary method not available. Skipping summary.
Model forward succeeded. Output shape: torch.Size([1, 2, 1200, 1200])
Exporting using torch.jit.script.
Model forward succeeded. Output shape: torch.Size([1, 2, 1200, 1200])
Exporting using torch.jit.script.


/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.9/site-packages/torch/onnx/utils.py:825: UserWarning: no signature found for <torch.ScriptMethod object at 0x7fbfdb4734f0>, skipping _decide_input_format
  warnings.warn(f"{e}, skipping _decide_input_format")
